# Linux for Data Engineers
## Essential Commands, Shell Scripting & Production Operations

**What you'll learn:**
- Linux file system structure and navigation
- Essential commands every DE uses daily
- File permissions and ownership (chmod, chown)
- Process management (ps, top, kill, nohup)
- Text processing power tools (grep, awk, sed, cut, sort, uniq)
- Shell scripting for pipeline automation
- Networking commands (curl, wget, ssh, scp, netstat)
- Cron jobs and scheduling
- Package management (apt, yum, pip)
- Docker essentials for DE
- Monitoring and troubleshooting
- Environment variables and configuration
- 50+ commands with real DE use cases

**Prerequisites:** None (beginner-friendly)
**Estimated Time:** 6-8 hours

---

> As a data engineer, you WILL work on Linux servers. Whether it's debugging
> a failed Spark job, deploying a pipeline, analyzing logs, or managing
> Docker containers — Linux skills are non-negotiable.

---

---
# Section 1: Linux File System & Navigation

## The Directory Tree

```
/                   <- Root (everything starts here)
├── home/           <- User home directories
│   └── dataeng/    <- Your home (~)
│       ├── projects/
│       ├── .bashrc
│       └── .ssh/
├── etc/            <- System configuration files
├── var/            <- Variable data (logs, databases)
│   └── log/       <- System and application logs
├── tmp/            <- Temporary files (cleared on reboot)
├── opt/            <- Optional/third-party software
├── usr/            <- User programs and libraries
│   ├── bin/       <- User commands
│   └── local/     <- Locally installed software
├── bin/            <- Essential system commands
├── proc/           <- Virtual filesystem (running processes info)
└── mnt/ or media/  <- Mount points for external storage
```

## Key Directories for Data Engineers

| Directory | What's There | DE Use Case |
|-----------|-------------|-------------|
| `/home/user/` | Your files | Scripts, configs, projects |
| `/var/log/` | Logs | Debug pipeline failures |
| `/tmp/` | Temp files | Staging data, intermediate files |
| `/opt/` | Software | Spark, Kafka, Airflow installations |
| `/etc/` | Config | Database configs, environment settings |
| `/mnt/` | Mounts | Network drives, cloud storage mounts |

In [ ]:
# Navigation and file system commands
# These are REFERENCE commands -- run them on a Linux terminal
# In Databricks, use %sh to run shell commands

print("FILE SYSTEM NAVIGATION")
print("=" * 60)

commands = {
    "Navigation": [
        ("pwd", "Print working directory (where am I?)"),
        ("ls", "List files in current directory"),
        ("ls -la", "List ALL files with details (permissions, size, date)"),
        ("ls -lh", "Human-readable file sizes (KB, MB, GB)"),
        ("ls -lt", "Sort by modification time (newest first)"),
        ("cd /path/to/dir", "Change directory"),
        ("cd ~", "Go to home directory"),
        ("cd -", "Go to previous directory"),
        ("cd ..", "Go up one level"),
    ],
    "File Operations": [
        ("cp source dest", "Copy file"),
        ("cp -r dir1/ dir2/", "Copy directory recursively"),
        ("mv old new", "Move or rename file/directory"),
        ("rm file.txt", "Delete file (CAREFUL -- no undo!)"),
        ("rm -rf directory/", "Delete directory recursively (DANGEROUS)"),
        ("mkdir -p path/to/dir", "Create nested directories"),
        ("touch file.txt", "Create empty file or update timestamp"),
        ("ln -s target link_name", "Create symbolic link"),
    ],
    "File Viewing": [
        ("cat file.txt", "Display entire file"),
        ("head -n 20 file.csv", "First 20 lines"),
        ("tail -n 50 file.log", "Last 50 lines"),
        ("tail -f /var/log/app.log", "Follow log in real-time (stream)"),
        ("less file.txt", "Paginated viewer (q to quit)"),
        ("wc -l file.csv", "Count lines in file"),
        ("du -sh directory/", "Disk usage of directory"),
        ("df -h", "Disk space on all mounted filesystems"),
    ],
}

for category, cmds in commands.items():
    print(f"\n  {category}:")
    print(f"  {'-' * 55}")
    for cmd, desc in cmds:
        print(f"    {cmd:<30} {desc}")

---
# Section 2: Text Processing — The DE Superpower

## grep, awk, sed, cut, sort, uniq

These commands let you analyze gigabytes of log files, CSV data, and
pipeline outputs without writing Python scripts. They're also MUCH faster
than Python for simple text operations (they're written in C).

## When to Use What

| Tool | Best For | Example |
|------|----------|---------|
| `grep` | Find lines matching a pattern | Find errors in logs |
| `awk` | Column-based processing | Extract specific fields from CSV |
| `sed` | Find and replace | Transform file contents |
| `cut` | Extract columns by delimiter | Quick column selection |
| `sort` | Sort lines | Order data |
| `uniq` | Deduplicate sorted lines | Count unique values |
| `tr` | Character translation | Replace delimiters |
| `xargs` | Build commands from input | Batch operations |

In [ ]:
# Text processing commands for DE
print("TEXT PROCESSING COMMANDS")
print("=" * 60)

examples = {
    "grep (search/filter)": [
        ("grep 'ERROR' app.log", "Find all error lines"),
        ("grep -i 'warning' app.log", "Case-insensitive search"),
        ("grep -c 'ERROR' app.log", "COUNT error lines"),
        ("grep -n 'FAILED' app.log", "Show line numbers"),
        ("grep -r 'password' /etc/", "Recursive search in directory"),
        ("grep -v 'DEBUG' app.log", "Exclude DEBUG lines (invert match)"),
        ("grep -E 'ERROR|WARN|FATAL' app.log", "Regex: multiple patterns"),
        ("zgrep 'ERROR' app.log.gz", "Search compressed files"),
    ],
    "awk (column processing)": [
        ("awk '{print $1, $3}' file.tsv", "Print columns 1 and 3"),
        ("awk -F',' '{print $2}' data.csv", "CSV: print 2nd column"),
        ("awk -F',' 'NR>1 {sum+=$3} END {print sum}' data.csv", "Sum column 3 (skip header)"),
        ("awk -F',' '$3 > 1000' orders.csv", "Filter rows where col3 > 1000"),
        ("awk -F',' '{print NR, $0}' file.csv", "Add line numbers"),
        ("awk 'END {print NR}' file.csv", "Count lines (like wc -l)"),
    ],
    "sed (find/replace)": [
        ("sed 's/old/new/g' file.txt", "Replace all 'old' with 'new'"),
        ("sed -i 's/localhost/prod-db/g' config.yml", "In-place edit"),
        ("sed -n '10,20p' file.txt", "Print lines 10-20"),
        ("sed '/^#/d' config.txt", "Delete comment lines"),
        ("sed '1d' file.csv", "Delete header (first line)"),
    ],
    "Pipelines (combining tools)": [
        ("cat app.log | grep 'ERROR' | wc -l", "Count errors"),
        ("cat data.csv | cut -d',' -f2 | sort | uniq -c | sort -rn | head", "Top values in column 2"),
        ("find /data/ -name '*.parquet' -mtime -1", "Find parquet files modified today"),
        ("du -sh /data/* | sort -rh | head -10", "Top 10 largest directories"),
        ("tail -f app.log | grep --line-buffered 'ERROR'", "Stream only errors"),
    ],
}

for category, cmds in examples.items():
    print(f"\n  {category}:")
    print(f"  {'-' * 60}")
    for cmd, desc in cmds:
        print(f"    {cmd}")
        print(f"      -> {desc}")

In [ ]:
# Real DE scenarios with text processing
print("\nREAL DATA ENGINEERING SCENARIOS")
print("=" * 60)

scenarios = [
    {
        "task": "Find which Spark jobs failed in the last hour",
        "command": "grep -E 'FAILED|ERROR' /var/log/spark/*.log | grep "$(date -d '1 hour ago' +%H:)"",
    },
    {
        "task": "Count records per partition in a CSV dump",
        "command": "awk -F',' '{print $5}' orders.csv | sort | uniq -c | sort -rn",
    },
    {
        "task": "Find large files consuming disk space",
        "command": "find /data/delta/ -name '*.parquet' -size +1G | xargs ls -lh",
    },
    {
        "task": "Check how many rows in a gzipped CSV (without decompressing fully)",
        "command": "zcat data.csv.gz | wc -l",
    },
    {
        "task": "Extract unique IP addresses from access log",
        "command": "awk '{print $1}' access.log | sort -u | wc -l",
    },
    {
        "task": "Monitor pipeline output file growth in real-time",
        "command": "watch -n 5 'ls -lh /output/data/ | tail -5'",
    },
    {
        "task": "Find all Python files that import pyspark",
        "command": "grep -rl 'from pyspark' --include='*.py' /projects/",
    },
    {
        "task": "Replace database hostname across all config files",
        "command": "find /etc/pipelines/ -name '*.yml' -exec sed -i 's/old-db/new-db/g' {} +",
    },
]

for i, s in enumerate(scenarios, 1):
    print(f"\n  Scenario {i}: {s['task']}")
    print(f"  $ {s['command']}")

---
# Section 3: File Permissions & Ownership

## Understanding Permission Strings

```
-rwxr-xr-- 1 dataeng pipelines 4096 Jun 15 10:30 pipeline.py
│└┬┘└┬┘└┬┘   └─owner─┘ └─group─┘ └size┘ └──date──┘ └─name─┘
│ │   │   │
│ │   │   └── Others: r-- (read only)
│ │   └────── Group:  r-x (read + execute)
│ └────────── Owner:  rwx (read + write + execute)
└──────────── File type: - (regular file), d (directory), l (symlink)
```

## Numeric (Octal) Permissions

| Number | Permission | Meaning |
|--------|-----------|---------|
| 7 | rwx | Read + Write + Execute |
| 6 | rw- | Read + Write |
| 5 | r-x | Read + Execute |
| 4 | r-- | Read only |
| 0 | --- | No permissions |

Common patterns:
- `755` = Owner:rwx, Group:r-x, Others:r-x (scripts, directories)
- `644` = Owner:rw-, Group:r--, Others:r-- (config files)
- `600` = Owner:rw-, Group:---, Others:--- (secrets, SSH keys)
- `777` = Everyone:rwx (DANGEROUS -- avoid in production)

In [ ]:
# Permissions commands
print("FILE PERMISSIONS")
print("=" * 60)

perm_commands = [
    ("chmod 755 script.sh", "Make script executable by all, writable by owner"),
    ("chmod 600 secrets.env", "Only owner can read/write (for credentials!)"),
    ("chmod +x deploy.sh", "Add execute permission for owner"),
    ("chmod -R 755 /opt/pipeline/", "Recursively set permissions"),
    ("chown dataeng:pipelines file.py", "Change owner and group"),
    ("chown -R dataeng:dataeng /home/dataeng/", "Recursive ownership change"),
    ("umask 022", "Set default permissions for new files (644 files, 755 dirs)"),
]

print("  Permission commands:")
for cmd, desc in perm_commands:
    print(f"    {cmd:<45} {desc}")

print("\n  DE Security Rules:")
print("    1. SSH keys: chmod 600 ~/.ssh/id_rsa (MUST be 600)")
print("    2. .env files: chmod 600 (contain secrets)")
print("    3. Scripts: chmod 755 (everyone can execute, only owner can edit)")
print("    4. Data dirs: chmod 775 (group can write, others can read)")
print("    5. NEVER chmod 777 in production (security vulnerability)")

---
# Section 4: Process Management

## Monitoring and Controlling Running Processes

As a DE, you'll need to:
- Find hung Spark processes
- Kill jobs that are consuming too much memory
- Run long pipelines in the background
- Monitor system resource usage

In [ ]:
# Process management commands
print("PROCESS MANAGEMENT")
print("=" * 60)

process_commands = {
    "Viewing Processes": [
        ("ps aux", "Show all running processes"),
        ("ps aux | grep spark", "Find Spark processes"),
        ("ps aux | grep python | grep -v grep", "Find Python processes"),
        ("top", "Interactive process monitor (q to quit)"),
        ("htop", "Better interactive monitor (install with apt/yum)"),
        ("pgrep -f 'spark-submit'", "Get PIDs of Spark jobs"),
    ],
    "Killing Processes": [
        ("kill PID", "Graceful terminate (SIGTERM)"),
        ("kill -9 PID", "Force kill (SIGKILL) -- use as last resort"),
        ("pkill -f 'pipeline.py'", "Kill by name pattern"),
        ("killall python3", "Kill all Python processes (CAREFUL!)"),
    ],
    "Background & Foreground": [
        ("command &", "Run command in background"),
        ("nohup python pipeline.py &", "Run even after logout (output to nohup.out)"),
        ("nohup python etl.py > /var/log/etl.log 2>&1 &", "Background with custom log"),
        ("jobs", "List background jobs"),
        ("fg %1", "Bring job 1 to foreground"),
        ("Ctrl+Z then bg", "Suspend current job, then run in background"),
        ("disown %1", "Detach job from terminal (survives logout)"),
    ],
    "Resource Monitoring": [
        ("free -h", "Memory usage (total, used, available)"),
        ("df -h", "Disk space per filesystem"),
        ("du -sh /data/*", "Disk usage per directory"),
        ("iostat -x 1", "Disk I/O statistics (per second)"),
        ("vmstat 1", "Virtual memory statistics"),
        ("uptime", "System load average (1, 5, 15 min)"),
        ("lsof -i :8080", "What process is using port 8080?"),
    ],
}

for category, cmds in process_commands.items():
    print(f"\n  {category}:")
    for cmd, desc in cmds:
        print(f"    {cmd:<50} {desc}")

---
# Section 5: Shell Scripting for Pipeline Automation

## Why Shell Scripts Matter for DE

- Automate deployment (deploy.sh, rollback.sh)
- Wrapper scripts for spark-submit
- Health checks and monitoring
- Data file management (rotate, archive, compress)
- Cron job scripts
- CI/CD pipeline steps

In [ ]:
# Shell scripting patterns for DE
print("SHELL SCRIPTING PATTERNS")
print("=" * 60)

scripts = {
    "1. Basic Pipeline Runner": """#!/bin/bash
# run_pipeline.sh -- Execute daily ETL pipeline
set -euo pipefail  # Exit on error, undefined vars, pipe failures

# Configuration
ENV="${ENV:-production}"  # Default to production
DATE="${1:-$(date -d yesterday +%Y-%m-%d)}"  # Arg or yesterday
LOG_DIR="/var/log/pipelines"
LOG_FILE="${LOG_DIR}/etl_${DATE}.log"

echo "[$(date)] Starting ETL for ${DATE} (env: ${ENV})" | tee -a "$LOG_FILE"

# Run Spark job
spark-submit \
  --master yarn \
  --deploy-mode cluster \
  --num-executors 20 \
  --executor-memory 8g \
  --conf spark.sql.shuffle.partitions=200 \
  /opt/pipelines/daily_etl.py \
  --date "$DATE" \
  --env "$ENV" \
  2>&1 | tee -a "$LOG_FILE"

EXIT_CODE=$?
if [ $EXIT_CODE -eq 0 ]; then
    echo "[$(date)] ETL completed successfully" | tee -a "$LOG_FILE"
else
    echo "[$(date)] ETL FAILED with exit code $EXIT_CODE" | tee -a "$LOG_FILE"
    # Send alert (email, Slack, PagerDuty)
    curl -X POST "$SLACK_WEBHOOK" -d "{\"text\": \"ETL FAILED for $DATE\"}"
    exit $EXIT_CODE
fi""",

    "2. Data Quality Check": """#!/bin/bash
# check_data.sh -- Validate pipeline output before promoting
set -euo pipefail

TABLE="$1"
MIN_ROWS="${2:-1000}"

echo "Checking table: $TABLE (minimum rows: $MIN_ROWS)"

# Count rows
ROW_COUNT=$(spark-sql -e "SELECT COUNT(*) FROM $TABLE" 2>/dev/null | tail -1)

if [ "$ROW_COUNT" -lt "$MIN_ROWS" ]; then
    echo "FAIL: $TABLE has only $ROW_COUNT rows (need $MIN_ROWS)"
    exit 1
fi

# Check for nulls in key column
NULL_COUNT=$(spark-sql -e "SELECT COUNT(*) FROM $TABLE WHERE id IS NULL" 2>/dev/null | tail -1)

if [ "$NULL_COUNT" -gt 0 ]; then
    echo "FAIL: $TABLE has $NULL_COUNT null IDs"
    exit 1
fi

echo "PASS: $TABLE has $ROW_COUNT rows, 0 null IDs"
""",

    "3. Log Rotation & Cleanup": """#!/bin/bash
# cleanup.sh -- Rotate logs and clean old data
set -euo pipefail

LOG_DIR="/var/log/pipelines"
DATA_DIR="/data/staging"
RETENTION_DAYS=30

echo "Cleaning files older than $RETENTION_DAYS days..."

# Compress logs older than 1 day
find "$LOG_DIR" -name "*.log" -mtime +1 -exec gzip {} \;

# Delete compressed logs older than retention
DELETED=$(find "$LOG_DIR" -name "*.gz" -mtime +$RETENTION_DAYS -delete -print | wc -l)
echo "  Deleted $DELETED old log files"

# Clean staging data older than 7 days
STAGING_DELETED=$(find "$DATA_DIR" -type f -mtime +7 -delete -print | wc -l)
echo "  Deleted $STAGING_DELETED staging files"

# Report disk usage
echo "  Current disk usage:"
df -h /data | tail -1 | awk '{print "    " $5 " used (" $4 " available)"}'
""",
}

for title, script in scripts.items():
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    for line in script.strip().split("\n")[:15]:
        print(f"  {line}")
    if script.count("\n") > 15:
        print(f"  ... ({script.count(chr(10)) - 15} more lines)")

In [ ]:
# Essential Bash constructs
print("\nESSENTIAL BASH CONSTRUCTS")
print("=" * 60)

constructs = """
# Variables
NAME="pipeline"
DATE=$(date +%Y-%m-%d)     # Command substitution
COUNT=${COUNT:-0}           # Default value if not set

# Conditionals
if [ -f "$FILE" ]; then     # File exists?
    echo "Processing $FILE"
elif [ -d "$DIR" ]; then    # Directory exists?
    echo "Scanning $DIR"
else
    echo "Not found" && exit 1
fi

# Loops
for file in /data/raw/*.csv; do
    echo "Processing $file"
    spark-submit process.py --input "$file"
done

# While loop (read lines from file)
while IFS= read -r line; do
    echo "Table: $line"
done < tables.txt

# Functions
validate_table() {
    local table="$1"
    local count=$(spark-sql -e "SELECT COUNT(*) FROM $table" | tail -1)
    [ "$count" -gt 0 ] && return 0 || return 1
}

# Error handling
set -euo pipefail           # Strict mode (ALWAYS use this)
trap 'echo "Error on line $LINENO"; exit 1' ERR  # Catch errors

# Useful tests
[ -f file ]     # File exists
[ -d dir ]      # Directory exists
[ -s file ]     # File is non-empty
[ -z "$var" ]   # Variable is empty
[ -n "$var" ]   # Variable is non-empty
[ "$a" = "$b" ] # String equality
[ "$a" -gt 5 ]  # Numeric comparison
"""

for line in constructs.strip().split("\n"):
    print(f"  {line}")

---
# Section 6: Networking Commands

## SSH, File Transfer, HTTP, and Port Management

Data engineers constantly:
- SSH into production servers
- Transfer files between systems
- Call APIs for data ingestion
- Debug network connectivity issues

In [ ]:
# Networking commands
print("NETWORKING FOR DATA ENGINEERS")
print("=" * 60)

network_commands = {
    "SSH (Secure Shell)": [
        ("ssh user@host", "Connect to remote server"),
        ("ssh -i key.pem user@host", "Connect with private key"),
        ("ssh -L 8080:localhost:8080 user@host", "Port forwarding (access remote Spark UI locally)"),
        ("ssh -J jumphost user@target", "Connect via jump/bastion host"),
        ("ssh-keygen -t ed25519", "Generate SSH key pair"),
        ("ssh-copy-id user@host", "Install public key on remote server"),
    ],
    "File Transfer": [
        ("scp file.py user@host:/path/", "Copy file to remote"),
        ("scp user@host:/path/data.csv ./", "Copy file from remote"),
        ("scp -r directory/ user@host:/path/", "Copy directory recursively"),
        ("rsync -avz /data/ user@host:/backup/", "Sync directories (incremental, compressed)"),
        ("rsync --progress large_file.parquet user@host:/data/", "Transfer with progress bar"),
    ],
    "HTTP / API Calls": [
        ("curl https://api.example.com/data", "GET request"),
        ("curl -X POST -H 'Content-Type: application/json' -d '{...}' URL", "POST with JSON"),
        ("curl -o output.json https://api.example.com/data", "Save response to file"),
        ("curl -s URL | python3 -m json.tool", "Pretty-print JSON response"),
        ("wget -q https://example.com/data.csv", "Download file quietly"),
    ],
    "Network Diagnostics": [
        ("ping host", "Check if host is reachable"),
        ("nc -zv host 5432", "Check if port is open (PostgreSQL)"),
        ("netstat -tlnp", "Show listening ports"),
        ("ss -tlnp", "Modern alternative to netstat"),
        ("curl -I https://example.com", "Check HTTP headers only"),
        ("nslookup hostname", "DNS lookup"),
        ("traceroute host", "Show network path to host"),
    ],
}

for category, cmds in network_commands.items():
    print(f"\n  {category}:")
    for cmd, desc in cmds:
        print(f"    {cmd}")
        print(f"      -> {desc}")

---
# Section 7: Cron Jobs & Scheduling

## Automating Pipeline Execution

Cron is the original scheduler (before Airflow existed). Still used for:
- Simple periodic tasks (log rotation, backups)
- Triggering pipeline runners
- Health checks and monitoring
- File cleanup

## Cron Expression Format

```
* * * * * command
│ │ │ │ │
│ │ │ │ └── Day of week (0-7, 0=Sunday)
│ │ │ └──── Month (1-12)
│ │ └────── Day of month (1-31)
│ └──────── Hour (0-23)
└────────── Minute (0-59)

Examples:
0 * * * *        Every hour (at minute 0)
0 2 * * *        Daily at 2:00 AM
0 2 * * 1-5      Weekdays at 2:00 AM
*/15 * * * *     Every 15 minutes
0 0 1 * *        First day of every month at midnight
30 6 * * 1       Every Monday at 6:30 AM
```

In [ ]:
# Cron job examples for DE
print("CRON JOBS FOR DATA ENGINEERS")
print("=" * 60)

cron_examples = [
    ("0 2 * * * /opt/pipelines/run_daily_etl.sh", "Daily ETL at 2 AM"),
    ("0 */4 * * * /opt/scripts/check_data_freshness.sh", "Check data freshness every 4 hours"),
    ("30 1 * * * /opt/scripts/cleanup_staging.sh", "Clean staging at 1:30 AM"),
    ("0 6 * * 1 /opt/scripts/weekly_report.sh", "Weekly report Monday 6 AM"),
    ("*/5 * * * * /opt/monitoring/health_check.sh", "Health check every 5 minutes"),
    ("0 0 1 * * /opt/scripts/monthly_archive.sh", "Monthly archive on the 1st"),
    ("0 3 * * * find /tmp -mtime +7 -delete", "Delete temp files older than 7 days"),
]

print("  Example crontab entries:")
print(f"  {'Cron Expression':<55} {'Purpose'}")
print(f"  {'-'*80}")
for cron, desc in cron_examples:
    print(f"  {cron:<55} {desc}")

print("\n  Cron management commands:")
print("    crontab -e        Edit your cron jobs")
print("    crontab -l        List your cron jobs")
print("    crontab -r        Remove all your cron jobs (CAREFUL!)")
print("    systemctl status cron   Check if cron service is running")

print("\n  Pro tips:")
print("    1. Always redirect output: command >> /var/log/cron.log 2>&1")
print("    2. Use full paths (cron has minimal PATH)")
print("    3. Test scripts manually before scheduling")
print("    4. For complex workflows, use Airflow instead of cron")

---
# Section 8: Environment Variables & Package Management

## Configuration Without Hardcoding

Environment variables are THE way to configure pipelines across environments
(dev, staging, production) without changing code.

In [ ]:
# Environment variables
print("ENVIRONMENT VARIABLES")
print("=" * 60)

env_commands = [
    ("export DB_HOST=prod-db.company.com", "Set variable for current session"),
    ("echo $DB_HOST", "Read variable"),
    ("env | grep DB_", "List all env vars matching pattern"),
    ("unset DB_HOST", "Remove variable"),
    ("source .env", "Load variables from file"),
    ("printenv", "Print all environment variables"),
]

print("  Commands:")
for cmd, desc in env_commands:
    print(f"    {cmd:<45} {desc}")

print("\n  .env file pattern (load with 'source .env'):")
env_file = """
    # .env -- Pipeline configuration
    export ENV=production
    export DB_HOST=prod-db.internal
    export DB_PORT=5432
    export DB_USER=pipeline_user
    export DB_PASSWORD=secret123  # In practice, use a secret manager!
    export SPARK_HOME=/opt/spark
    export WAREHOUSE_PATH=s3://company-lake/warehouse/
    export LOG_LEVEL=INFO
"""
print(env_file)

print("  In Python:")
print("    import os")
print("    db_host = os.environ.get('DB_HOST', 'localhost')  # With default")
print("    db_pass = os.environ['DB_PASSWORD']  # Raises KeyError if missing")

print("\n  Security rule: NEVER commit .env files to git!")
print("  Add to .gitignore: .env, *.env, secrets/")

In [ ]:
# Package management
print("\nPACKAGE MANAGEMENT")
print("=" * 60)

pkg_commands = {
    "apt (Ubuntu/Debian)": [
        ("sudo apt update", "Update package index"),
        ("sudo apt install package-name", "Install package"),
        ("sudo apt upgrade", "Upgrade all packages"),
        ("apt list --installed | grep python", "List installed Python-related"),
        ("dpkg -l | grep java", "Check if Java is installed"),
    ],
    "yum/dnf (RHEL/CentOS/Amazon Linux)": [
        ("sudo yum install package-name", "Install package"),
        ("sudo yum update", "Update all packages"),
        ("yum list installed | grep spark", "Check installed packages"),
    ],
    "pip (Python)": [
        ("pip install package==1.2.3", "Install specific version"),
        ("pip install -r requirements.txt", "Install from requirements file"),
        ("pip freeze > requirements.txt", "Export installed packages"),
        ("pip list --outdated", "Show packages needing update"),
        ("python -m venv .venv && source .venv/bin/activate", "Create virtual environment"),
    ],
}

for category, cmds in pkg_commands.items():
    print(f"\n  {category}:")
    for cmd, desc in cmds:
        print(f"    {cmd:<55} {desc}")

---
# Section 9: Docker Essentials for DE

## Why Docker Matters for Data Engineers

- **Reproducible environments** (works on my machine → works everywhere)
- **Isolate dependencies** (Python 3.8 for one pipeline, 3.11 for another)
- **Deploy pipelines** (package code + deps into a container)
- **Local development** (run Kafka, Postgres, Airflow locally)
- **CI/CD** (build and test in containers)

In [ ]:
# Docker commands for DE
print("DOCKER ESSENTIALS")
print("=" * 60)

docker_commands = {
    "Container Management": [
        ("docker run -it python:3.11 bash", "Start Python container interactively"),
        ("docker run -d -p 8080:8080 airflow", "Run Airflow in background, expose port"),
        ("docker ps", "List running containers"),
        ("docker ps -a", "List ALL containers (including stopped)"),
        ("docker stop container_id", "Stop a container"),
        ("docker rm container_id", "Remove a container"),
        ("docker logs container_id", "View container logs"),
        ("docker exec -it container_id bash", "Shell into running container"),
    ],
    "Image Management": [
        ("docker build -t my-pipeline:v1 .", "Build image from Dockerfile"),
        ("docker images", "List local images"),
        ("docker pull postgres:15", "Download image from registry"),
        ("docker tag my-app:v1 registry/my-app:v1", "Tag for pushing to registry"),
        ("docker push registry/my-app:v1", "Push to container registry"),
    ],
    "Docker Compose (multi-container)": [
        ("docker-compose up -d", "Start all services defined in docker-compose.yml"),
        ("docker-compose down", "Stop and remove all services"),
        ("docker-compose logs -f", "Follow logs from all services"),
        ("docker-compose ps", "Status of all services"),
    ],
}

for category, cmds in docker_commands.items():
    print(f"\n  {category}:")
    for cmd, desc in cmds:
        print(f"    {cmd:<55} {desc}")

print("\n  Example docker-compose.yml for local DE development:")
compose = """
    services:
      postgres:
        image: postgres:15
        environment:
          POSTGRES_PASSWORD: dev_password
        ports: ["5432:5432"]
      
      kafka:
        image: confluentinc/cp-kafka:latest
        ports: ["9092:9092"]
      
      airflow:
        image: apache/airflow:2.8.0
        ports: ["8080:8080"]
"""
print(compose)

---
# Section 10: Troubleshooting & Production Operations

## The DE Debugging Toolkit

When a pipeline fails at 3 AM, you need to diagnose FAST.

In [ ]:
# Troubleshooting playbook
print("TROUBLESHOOTING PLAYBOOK")
print("=" * 60)

playbook = {
    "Pipeline Failed -- First Steps": [
        "1. Check the logs: tail -100 /var/log/pipeline/etl.log",
        "2. Check error code: echo $?  (0=success, non-zero=failure)",
        "3. Check disk space: df -h  (full disk = common cause)",
        "4. Check memory: free -h  (OOM killer?)",
        "5. Check if service is running: systemctl status spark-master",
    ],
    "Spark Job OOM (Out of Memory)": [
        "1. Check Spark UI: executor memory usage",
        "2. Look for data skew: one partition much larger than others",
        "3. Solutions:",
        "   - Increase executor memory: --executor-memory 16g",
        "   - Reduce data skew: repartition/salt join keys",
        "   - Cache less: .unpersist() when done",
        "   - Reduce shuffle partitions for small data",
    ],
    "Disk Full": [
        "1. Find large files: du -sh /* | sort -rh | head",
        "2. Find large log files: find /var/log -size +100M",
        "3. Clean: journalctl --vacuum-size=100M  (system logs)",
        "4. Clean: docker system prune -a  (Docker artifacts)",
        "5. Clean: rm -rf /tmp/* (temporary files)",
    ],
    "Network/Connection Issues": [
        "1. DNS resolution: nslookup hostname",
        "2. Port reachable: nc -zv host port",
        "3. Firewall: sudo iptables -L -n",
        "4. Check route: traceroute host",
        "5. Test HTTP: curl -v https://api.example.com",
    ],
    "Slow Pipeline Performance": [
        "1. Check IO: iostat -x 1 5  (disk bottleneck?)",
        "2. Check CPU: top  (CPU-bound?)",
        "3. Check network: iftop  (network bottleneck?)",
        "4. Check Spark UI: shuffle read/write sizes",
        "5. Check data locality: are tasks running on data nodes?",
    ],
}

for situation, steps in playbook.items():
    print(f"\n  {situation}:")
    for step in steps:
        print(f"    {step}")

---
# Summary

## Linux Commands Cheat Sheet for Data Engineers

| Category | Essential Commands |
|----------|-------------------|
| **Navigate** | `cd`, `ls -la`, `pwd`, `find`, `tree` |
| **Files** | `cp`, `mv`, `rm`, `mkdir -p`, `touch`, `ln -s` |
| **View** | `cat`, `head`, `tail -f`, `less`, `wc -l` |
| **Search** | `grep -r`, `find`, `locate`, `which` |
| **Text** | `awk`, `sed`, `cut`, `sort`, `uniq`, `tr` |
| **Permissions** | `chmod`, `chown`, `umask` |
| **Processes** | `ps aux`, `top`, `kill`, `nohup`, `&` |
| **Disk** | `df -h`, `du -sh`, `ncdu` |
| **Network** | `ssh`, `scp`, `rsync`, `curl`, `nc` |
| **Scheduling** | `crontab -e`, `at`, `systemctl` |
| **Docker** | `docker run`, `docker-compose up`, `docker logs` |
| **Packages** | `apt install`, `pip install`, `pip freeze` |

## Top 10 Commands You'll Use Daily

1. `grep 'ERROR' /var/log/pipeline/*.log` — Find failures
2. `tail -f app.log` — Watch logs in real-time
3. `df -h` — Check disk space
4. `ps aux | grep spark` — Find running jobs
5. `ssh -L 4040:localhost:4040 server` — Access remote Spark UI
6. `rsync -avz /data/ backup:/data/` — Sync data
7. `find /data -name '*.parquet' -mtime -1` — Recent files
8. `du -sh /data/* | sort -rh | head` — What's using space
9. `kill -9 $(pgrep -f 'stuck_job')` — Kill hung process
10. `source .env && python3 pipeline.py` — Run with config

---
*Linux for Data Engineers — Essential commands and production operations*